# matrixlayout Binder demo

`matrixlayout` builds LaTeX/TikZ layouts for matrix-centered linear algebra figures and renders them to SVG. The examples below are adapted from the `la_figures` Gaussian-elimination, QR/Gram-Schmidt, eigenproblem, and back-substitution examples, but are written directly against the current `matrixlayout` API.

In [ ]:
from matrixlayout import (
    backsubst_svg,
    decorator_bg,
    decorator_box,
    decorator_color,
    render_eig_svg,
    render_ge_svg,
    render_qr_svg,
    sel_col,
    sel_entry,
    show_svg,
)

RENDER_OPTS = dict(toolchain_name="pdftex_dvisvgm", crop="tight", padding=(2, 2, 2, 2))

## Row reduction of an augmented system

This mirrors the computational layout produced by `ShowGE{Rational{Int}}(A, b); ref!(..., gj=false); show_layout!(..., fig_scale=1.2)` for a 3-by-5 system.

In [ ]:
A_aug = [
    [-2, 0, -2, 2, 0, 4],
    [2, 0, 3, -3, 0, -5],
    [0, 0, 0, 0, 1, -1],
]
E = [[1, 0, 0], [1, 1, 0], [0, 0, 1]]
U_aug = [
    [-2, 0, -2, 2, 0, 4],
    [0, 0, 1, -1, 0, -1],
    [0, 0, 0, 0, 1, -1],
]
U = [row[:-1] for row in U_aug]

row_reduction_svg = render_ge_svg(
    matrices=[[None, A_aug], [E, U_aug]],
    label_rows=[
        {"grid": (0, 1), "side": "above", "rows": [["$A$", "", "", "", "", "$b$"]]},
        {"grid": (1, 0), "side": "above", "rows": [["$E$"]]},
        {"grid": (1, 1), "side": "above", "rows": [["$U$", "", "", "", "", "$c$"]]},
    ],
    decorations=[
        {"grid": (0, 1), "vlines": [5]},
        {"grid": (1, 1), "vlines": [5]},
    ],
    fig_scale=1.2,
    outer_hspace_mm=6,
    **RENDER_OPTS,
)
show_svg(row_reduction_svg)

## Back-substitution for the same system

This mirrors `show_backsubstitution!(pb_ge, b_col=1)`: it displays the substitution cascade from the REF system, with free variables boxed first.

In [ ]:
show_backsubstitution_svg = backsubst_svg(
    cascade_txt=[
        r"{\ShortCascade%",
        r"{\ShortCascade%",
        r"{\ShortCascade%",
        r"{$\boxed{ x_2 = \alpha_2,\;x_4 = \alpha_4 }$}%",
        r"{$x_5 = -1$}%",
        r"{$\;\Rightarrow\;\boxed{x_5 = -1}$}%",
        r"}%",
        r"{$x_3 = x_{4} - 1$}%",
        r"{$\;\Rightarrow\;\boxed{x_3 = \alpha_{4} - 1}$}%",
        r"}%",
        r"{$x_1 = - \frac{1}{2} \left( 2 x_{3} - 2 x_{4} + 4 \right)$}%",
        r"{$\;\Rightarrow\;\boxed{x_1 = -1}$}%",
        r"}%",
    ],
    show_system=False,
    show_cascade=True,
    show_solution=False,
    **RENDER_OPTS,
)
show_svg(show_backsubstitution_svg)

## Pivot columns, equation labels, and callouts

The same reduced system is annotated with row and column labels. This is the kind of figure used to discuss pivot variables, free variables, and consistency.

In [ ]:
label_svg = render_ge_svg(
    matrices=[U],
    label_rows=[{"grid": (0, 0), "side": "above", "rows": [["$x_1$", "$x_2$", "$x_3$", "$x_4$", "$x_5$"]]}],
    label_cols=[{"grid": (0, 0), "side": "left", "cols": [["$p_1$"], ["$p_2$"], ["$p_3$"]]}],
    decorations=[
        {"grid": (0, 0), "entries": [sel_col(0)], "decorator": decorator_bg("green!15")},
        {"grid": (0, 0), "entries": [sel_col(2)], "decorator": decorator_bg("green!15")},
        {"grid": (0, 0), "entries": [sel_col(4)], "decorator": decorator_bg("green!15")},
        {"grid": (0, 0), "entries": [sel_entry(0, 0), sel_entry(1, 2), sel_entry(2, 4)], "decorator": decorator_box(color="red")},
        {"grid": (0, 0), "label": "x2 and x4 are free", "side": "right", "angle": -35, "length": 10},
    ],
    create_medium_nodes=True,
    **RENDER_OPTS,
)
show_svg(label_svg)

## Entry decorators for elimination structure

Selectors and decorators make it possible to call out pivots, eliminated entries, and active columns without manually rewriting the matrix.

In [ ]:
decorated_svg = render_ge_svg(
    matrices=[U],
    decorations=[
        {"grid": (0, 0), "entries": [sel_entry(0, 0), sel_entry(1, 2), sel_entry(2, 4)], "decorator": decorator_box(color="blue")},
        {"grid": (0, 0), "entries": [sel_entry(1, 0), sel_entry(2, 0), sel_entry(2, 2)], "decorator": decorator_color("gray")},
        {"grid": (0, 0), "entries": [sel_col(1), sel_col(3)], "decorator": decorator_bg("yellow!25")},
    ],
    **RENDER_OPTS,
)
show_svg(decorated_svg)

## Block-partitioned matrix layout

Matrixlayout can present a block matrix as a structured object, with partitions and labels attached to the same figure. This example shows an upper block-triangular matrix with blocks `B`, `C`, and `D`.

In [ ]:
block_matrix = [
    [1, 2, 0, 3],
    [0, 1, 0, -1],
    [0, 0, 4, 5],
    [0, 0, 0, 2],
]

block_svg = render_ge_svg(
    matrices=[block_matrix],
    label_rows=[
        {"grid": (0, 0), "side": "above", "rows": [["$B$", "", "$C$", ""]]},
    ],
    label_cols=[
        {"grid": (0, 0), "side": "left", "cols": [["$B$"], [""], ["$0$"], ["$D$"]]},
    ],
    decorations=[
        {"grid": (0, 0), "vlines": [2]},
        {"grid": (0, 0), "hlines": [2]},
        {"grid": (0, 0), "entries": [sel_entry(2, 0), sel_entry(2, 1), sel_entry(3, 0), sel_entry(3, 1)], "decorator": decorator_color("gray")},
    ],
    **RENDER_OPTS,
)
show_svg(block_svg)

## Side-by-side derived matrices

A single grid can compose related matrices without implying that matrixlayout computed them. This figure places `A`, `A^T A`, and `A^T b` together as the pieces of a least-squares normal-equation setup.

In [ ]:
A_ls = [[1, 0], [1, 1], [1, 2]]
AtA = [[3, 3], [3, 5]]
Atb = [[5], [6]]

comparison_svg = render_ge_svg(
    matrices=[[A_ls, AtA, Atb]],
    label_rows=[
        {"grid": (0, 0), "side": "above", "rows": [["$A$"]]},
        {"grid": (0, 1), "side": "above", "rows": [["$A^T A$"]]},
        {"grid": (0, 2), "side": "above", "rows": [["$A^T b$"]]},
    ],
    outer_hspace_mm=10,
    **RENDER_OPTS,
)
show_svg(comparison_svg)

## Gram-Schmidt / QR layout

This mirrors `h, mats = nM.gram_schmidt_qr(A; fig_scale=1.2); display(h)` for `A = [-1 -4 -3; 1 0 -3; 1 4 1; -1 0 1]`.

In [ ]:
A_qr = [
    [-1, -4, -3],
    [1, 0, -3],
    [1, 4, 1],
    [-1, 0, 1],
]
W = [
    [-1, -2, -1],
    [1, -2, -1],
    [1, 2, -1],
    [-1, 2, -1],
]
Wt = [
    [-1, 1, 1, -1],
    [-2, -2, 2, 2],
    [-1, -1, -1, -1],
]
WtA = [[4, 8, 0], [0, 16, 16], [0, 0, 4]]
WtW = [[4, 0, 0], [0, 16, 0], [0, 0, 4]]
S = [[(1, 2), 0, 0], [0, (1, 4), 0], [0, 0, (1, 2)]]
Qt = [
    [(-1, 2), (1, 2), (1, 2), (-1, 2)],
    [(-1, 2), (-1, 2), (1, 2), (1, 2)],
    [(-1, 2), (-1, 2), (-1, 2), (-1, 2)],
]
R = [[2, 4, 0], [0, 4, 4], [0, 0, 2]]

qr_svg = render_qr_svg(
    matrices=[[None, None, A_qr, W], [None, Wt, WtA, WtW], [S, Qt, R, None]],
    array_names=True,
    fig_scale=1.2,
    **RENDER_OPTS,
)
show_svg(qr_svg)

## Eigenproblem table

A compact table can show eigenvalues, eigenspace bases, and the eigenbasis matrix for a concrete diagonalizable matrix. Here `A = [[4, 1], [2, 3]]` has eigenvalues `5` and `2`.

In [ ]:
eig_spec = {
    "lambda": [5, 2],
    "ma": [1, 1],
    "evecs": [[[1, 1]], [[-1, 2]]],
    "qvecs": [[[1, 1]], [[-1, 2]]],
}

eig_svg = render_eig_svg(eig_spec, case="S", fig_scale=1.2, **RENDER_OPTS)
show_svg(eig_svg)

## SVD table

This mirrors `svg, spec = nM.show_svd_tbl(A, fig_scale=1.4); display(svg)` for a rank-one rational matrix.

In [ ]:
from fractions import Fraction as F

svd_spec = {
    "sigma": [1, 0],
    "lambda": [1, 0],
    "ma": [2, 2],
    "evecs": [
        [[F(-3, 4), 0, 1, 0], [0, 0, 0, 1]],
        [[0, 1, 0, 0], [F(4, 3), 0, 1, 0]],
    ],
    "qvecs": [
        [[F(-3, 5), 0, F(4, 5), 0], [0, 0, 0, 1]],
        [[0, 1, 0, 0], [F(4, 5), 0, F(3, 5), 0]],
    ],
    "uvecs": [
        [[F(-2, 7), F(-6, 7), F(-3, 7)], [F(3, 7), F(2, 7), F(-6, 7)]],
        [[F(6, 7), F(-3, 7), F(2, 7)]],
    ],
    "sz": (3, 4),
}

svd_svg = render_eig_svg(svd_spec, case="SVD", fig_scale=1.4, **RENDER_OPTS)
show_svg(svd_svg)

## Back-substitution trace

This mirrors the `la_figures` back-substitution notebook: render the system, the substitution cascade, and the parametric solution together.

In [ ]:
backsubst_demo_svg = backsubst_svg(
    system_txt=r"$x + 2y = 4,\quad y + 3z = 5$",
    cascade_txt=[r"$y = 5 - 3z$", r"$x = 4 - 2(5 - 3z) = 6z - 6$"],
    solution_txt=r"$(x,y,z) = (6t - 6,\; 5 - 3t,\; t)$",
    **RENDER_OPTS,
)
show_svg(backsubst_demo_svg)

## Render options

`toolchain_name`, `crop`, and `padding` are forwarded through the rendering boundary. This notebook uses `pdftex_dvisvgm` because it is compact and reliable in Binder.

In [ ]:
compact_svg = render_ge_svg(
    matrices=[[[1, 2], [3, 4]]],
    toolchain_name="pdftex_dvisvgm",
    crop="tight",
    padding=(0, 0, 0, 0),
)
show_svg(compact_svg)

## Optional: inspect SVG text

The rendered image is the useful output for most notebook work. Inspect raw SVG only when debugging, saving, or comparing renderer output.

In [ ]:
print(backsubst_demo_svg[:500])

## Troubleshooting

If rendering fails, check that the TeX packages and converters are available. The Binder image checks required TeX packages during `postBuild`, and the renderers keep failure artifacts in a temporary directory when a toolchain fails.

In [ ]:
import shutil

checks = ["latex", "dvisvgm", "kpsewhich"]
{name: shutil.which(name) for name in checks}